In [1]:
from pathlib import Path
import logging
import joblib
import numpy as np
import numpy.typing as npt
from tqdm import tqdm
from xgboost import XGBClassifier
from keras.models import load_model
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
)
import statsmodels.api as sm
from scipy.optimize import minimize_scalar, minimize
import matplotlib.pyplot as plt
from src.module.utils import load_config, get_peak_trough, process_meanbp

In [2]:
def brier_skill_score(y_true, y_prob):
    y_true = np.asarray(y_true, dtype=float)
    bs_model = brier_score_loss(y_true, y_prob)
    p = y_true.mean()  # 평가셋 자연 유병률
    bs_ref = p * (1.0 - p)  # 항상 p를 예측하는 무정보 모델
    return 1.0 - bs_model / bs_ref, bs_model, bs_ref


def scale_input(input) -> npt.NDArray:
    return input.astype(np.float32) / 200

In [3]:
config = load_config("../../config/conf.yaml")
src_path = Path(config["data_path"]).expanduser() / "mover" / "06.train_val_test_new"
seed1_path = Path(config["model_path"]).expanduser() / "revision_seed42"
seed2_path = Path(config["model_path"]).expanduser() / "revision_seed43"
seed3_path = Path(config["model_path"]).expanduser() / "revision_seed44"
strategy = "hypophetversion2"

In [4]:
test_cid = np.load(src_path / f"mover_{strategy}_test_cid.npy")
test_X = np.load(src_path / f"mover_{strategy}_test_X.npy")
test_y = np.load(src_path / f"mover_{strategy}_test_y.npy")
print(test_cid.shape, test_X.shape, test_y.shape)

(25793,) (25793, 3000) (25793,)


In [5]:
test_X_s = scale_input(test_X)

# CNN

In [6]:
cnn_seed42 = load_model(seed1_path / "cnn.model.keras")
cnn_seed43 = load_model(seed2_path / "cnn.model.keras")
cnn_seed44 = load_model(seed3_path / "cnn.model.keras")

2026-10-08 17:05:47.274256: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M5 Max
2026-10-08 17:05:47.274302: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 128.00 GB
2026-10-08 17:05:47.274309: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 53.76 GB
2026-10-08 17:05:47.274322: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-10-08 17:05:47.274330: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [7]:
prob_y_seed42 = cnn_seed42.predict(test_X_s).reshape(-1)
prob_y_seed43 = cnn_seed43.predict(test_X_s).reshape(-1)
prob_y_seed44 = cnn_seed44.predict(test_X_s).reshape(-1)

2026-10-08 17:05:49.655155: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


807/807 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
807/807 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
807/807 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


In [8]:
auroc_seed42 = roc_auc_score(test_y, prob_y_seed42)
bss_seed42, _, _ = brier_skill_score(test_y, prob_y_seed42)
auroc_seed43 = roc_auc_score(test_y, prob_y_seed43)
bss_seed43, _, _ = brier_skill_score(test_y, prob_y_seed43)
auroc_seed44 = roc_auc_score(test_y, prob_y_seed44)
bss_seed44, _, _ = brier_skill_score(test_y, prob_y_seed44)

In [10]:
print(
    f"AUROC: {np.mean([auroc_seed42, auroc_seed43, auroc_seed44]):.4f} ± {np.std([auroc_seed42, auroc_seed43, auroc_seed44]):.4f}"
)
print(
    f"BSS: {np.mean([bss_seed42, bss_seed43, bss_seed44]):.4f} ± {np.std([bss_seed42, bss_seed43, bss_seed44]):.4f}"
)

AUROC: 0.7332 ± 0.0009
BSS: 0.0780 ± 0.0066


# ResNet

In [6]:
resnet_seed42 = load_model(seed1_path / "resnet.model.keras")
resnet_seed43 = load_model(seed2_path / "resnet.model.keras")
resnet_seed44 = load_model(seed3_path / "resnet.model.keras")

2026-10-09 09:29:16.749346: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M5 Max
2026-10-09 09:29:16.749372: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 128.00 GB
2026-10-09 09:29:16.749378: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 53.76 GB
2026-10-09 09:29:16.749391: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-10-09 09:29:16.749399: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [7]:
prob_y_seed42 = resnet_seed42.predict(test_X_s).reshape(-1)
prob_y_seed43 = resnet_seed43.predict(test_X_s).reshape(-1)
prob_y_seed44 = resnet_seed44.predict(test_X_s).reshape(-1)

2026-10-09 09:29:19.069518: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


807/807 ━━━━━━━━━━━━━━━━━━━━ 22s 27ms/step
807/807 ━━━━━━━━━━━━━━━━━━━━ 22s 26ms/step
807/807 ━━━━━━━━━━━━━━━━━━━━ 22s 27ms/step


In [8]:
auroc_seed42 = roc_auc_score(test_y, prob_y_seed42)
bss_seed42, _, _ = brier_skill_score(test_y, prob_y_seed42)
auroc_seed43 = roc_auc_score(test_y, prob_y_seed43)
bss_seed43, _, _ = brier_skill_score(test_y, prob_y_seed43)
auroc_seed44 = roc_auc_score(test_y, prob_y_seed44)
bss_seed44, _, _ = brier_skill_score(test_y, prob_y_seed44)

In [9]:
print(
    f"AUROC: {np.mean([auroc_seed42, auroc_seed43, auroc_seed44]):.4f} ± {np.std([auroc_seed42, auroc_seed43, auroc_seed44]):.4f}"
)
print(
    f"BSS: {np.mean([bss_seed42, bss_seed43, bss_seed44]):.4f} ± {np.std([bss_seed42, bss_seed43, bss_seed44]):.4f}"
)

AUROC: 0.7362 ± 0.0010
BSS: -0.3970 ± 0.1166


# LSTMFCN

In [6]:
lstmfcn_seed42 = load_model(seed1_path / "lstmfcn.model.keras")
lstmfcn_seed43 = load_model(seed2_path / "lstmfcn.model.keras")
lstmfcn_seed44 = load_model(seed3_path / "lstmfcn.model.keras")

2026-10-09 09:31:16.113582: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M5 Max
2026-10-09 09:31:16.113611: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 128.00 GB
2026-10-09 09:31:16.113616: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 53.76 GB
2026-10-09 09:31:16.113633: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-10-09 09:31:16.113643: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [7]:
prob_y_seed42 = lstmfcn_seed42.predict(test_X_s).reshape(-1)
prob_y_seed43 = lstmfcn_seed43.predict(test_X_s).reshape(-1)
prob_y_seed44 = lstmfcn_seed44.predict(test_X_s).reshape(-1)

  5/807 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step  

2026-10-09 09:31:16.729516: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


807/807 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step
807/807 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step
807/807 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step


In [8]:
auroc_seed42 = roc_auc_score(test_y, prob_y_seed42)
bss_seed42, _, _ = brier_skill_score(test_y, prob_y_seed42)
auroc_seed43 = roc_auc_score(test_y, prob_y_seed43)
bss_seed43, _, _ = brier_skill_score(test_y, prob_y_seed43)
auroc_seed44 = roc_auc_score(test_y, prob_y_seed44)
bss_seed44, _, _ = brier_skill_score(test_y, prob_y_seed44)

In [9]:
print(
    f"AUROC: {np.mean([auroc_seed42, auroc_seed43, auroc_seed44]):.4f} ± {np.std([auroc_seed42, auroc_seed43, auroc_seed44]):.4f}"
)
print(
    f"BSS: {np.mean([bss_seed42, bss_seed43, bss_seed44]):.4f} ± {np.std([bss_seed42, bss_seed43, bss_seed44]):.4f}"
)

AUROC: 0.7199 ± 0.0025
BSS: -0.2030 ± 0.0115


# InceptionTime

In [6]:
inceptiontime_seed42 = load_model(seed1_path / "inceptiontime.model.keras")
inceptiontime_seed43 = load_model(seed2_path / "inceptiontime.model.keras")
inceptiontime_seed44 = load_model(seed3_path / "inceptiontime.model.keras")

2026-10-09 11:11:23.038422: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M5 Max
2026-10-09 11:11:23.038454: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 128.00 GB
2026-10-09 11:11:23.038459: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 53.76 GB
2026-10-09 11:11:23.038477: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-10-09 11:11:23.038488: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [7]:
prob_y_seed42 = inceptiontime_seed42.predict(test_X_s).reshape(-1)
prob_y_seed43 = inceptiontime_seed43.predict(test_X_s).reshape(-1)
prob_y_seed44 = inceptiontime_seed44.predict(test_X_s).reshape(-1)

2026-10-09 11:11:24.580181: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


807/807 ━━━━━━━━━━━━━━━━━━━━ 13s 16ms/step
807/807 ━━━━━━━━━━━━━━━━━━━━ 13s 16ms/step
807/807 ━━━━━━━━━━━━━━━━━━━━ 13s 16ms/step


In [8]:
auroc_seed42 = roc_auc_score(test_y, prob_y_seed42)
bss_seed42, _, _ = brier_skill_score(test_y, prob_y_seed42)
auroc_seed43 = roc_auc_score(test_y, prob_y_seed43)
bss_seed43, _, _ = brier_skill_score(test_y, prob_y_seed43)
auroc_seed44 = roc_auc_score(test_y, prob_y_seed44)
bss_seed44, _, _ = brier_skill_score(test_y, prob_y_seed44)

In [9]:
print(
    f"AUROC: {np.mean([auroc_seed42, auroc_seed43, auroc_seed44]):.4f} ± {np.std([auroc_seed42, auroc_seed43, auroc_seed44]):.4f}"
)
print(
    f"BSS: {np.mean([bss_seed42, bss_seed43, bss_seed44]):.4f} ± {np.std([bss_seed42, bss_seed43, bss_seed44]):.4f}"
)

AUROC: 0.7312 ± 0.0043
BSS: -0.2295 ± 0.0510
